In [1]:
suppressPackageStartupMessages(library(DESeq2))
library(DESeq2)

In [2]:

counts_matrix <- read.csv("/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/quantified/tximeta_gene_counts_matrix.csv", row.names = NULL)
cols_to_keep <- !grepl("KPC-2_LoA", colnames(counts_matrix))
counts_matrix <- counts_matrix[, cols_to_keep]

boss_low <- read.csv("/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/250716_met_low_genes_mouse_no_H10.csv", row.names = NULL)

boss_high <- read.csv("/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/250716_met_high_genes_mouse_no_H10.csv", row.names = NULL)


gene_names <- counts_matrix[, 1]
gene_names <- gsub("\\..*", "", gene_names)


unique_idx <- !duplicated(gene_names)


counts_matrix <- counts_matrix[unique_idx, ]
rownames(counts_matrix) <- gene_names[unique_idx]


counts_matrix <- counts_matrix[, -1]
counts_matrix <- round(counts_matrix)


boss_low_genes <- unique(gsub("\\..*", "", boss_low[, 1]))
boss_high_genes <- unique(gsub("\\..*", "", boss_high[, 1]))

Warning message in file(file, "rt"):
“cannot open file '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Reference/handler_2025/quantified/tximeta_gene_counts_matrix.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
sample_names <- colnames(counts_matrix)


parental_line <- sapply(strsplit(sample_names, "_"), `[`, 1)


metastasis <- ifelse(grepl("Hi", sample_names), "High", "Low")


coldata <- data.frame(
  row.names = sample_names,
  Parental_Line = as.factor(parental_line),
  Metastasis = as.factor(metastasis)
)

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = counts_matrix,
                              colData = coldata,
                              design = ~ Parental_Line + Metastasis)


keep <- rowSums(counts(dds)) > 50
dds_nonzero <- dds[keep,]


dds_nonzero$Metastasis <- relevel(dds_nonzero$Metastasis, ref = "Low")


diff_exp <- DESeq(dds_nonzero)


res <- results(diff_exp, alpha = 0.05)
resSig <- subset(res, padj < 0.05)

In [ ]:
my_high_genes <- rownames(subset(resSig, log2FoldChange > 0))
my_low_genes  <- rownames(subset(resSig, log2FoldChange < 0))

boss_low_genes <- unique(gsub("\\..*", "", boss_low[, 1]))
boss_high_genes <- unique(gsub("\\..*", "", boss_high[, 1]))


high_overlap <- intersect(my_high_genes, boss_high_genes)
low_overlap  <- intersect(my_low_genes, boss_low_genes)


print("========== Verifying Results ==========")
print(paste("My Met-High genes:", length(my_high_genes)))
print(paste("PI's Met-High genes:", length(boss_high_genes)))
print(paste("Matching Met-High genes:", length(high_overlap)))
print("--------------------------------")
print(paste("My Met-Low genes:", length(my_low_genes)))
print(paste("PI's Met-Low genes:", length(boss_low_genes)))
print(paste("Matching Met-Low genes:", length(low_overlap)))

In [ ]:

head(my_high_genes)

head(boss_high_genes)